# Hybrid Engine — Integration Tests

**Doctor's deterministic routing + Claude Haiku LLM extraction**

The doctor's `IntakeEngine` drives all routing:
phase ordering, parent/child gating, ask_if/skip_if logic,
budget enforcement, concept deduplication, red flag detection.

`HaikuAnswerExtractor` replaces keyword-based `normalize_boolean()`
with Claude Haiku free-text understanding.

Run from `CAPSTONE2026_SPRING/` root.

In [1]:
import importlib, sys
sys.path.insert(0, '.')

import hybrid_engine
importlib.reload(hybrid_engine)
import hybrid_engine.extractor, hybrid_engine.session
importlib.reload(hybrid_engine.extractor)
importlib.reload(hybrid_engine.session)

from hybrid_engine import HybridIntakeSession

print('Available complaints:')
for c in HybridIntakeSession.available_complaints():
    print(f"  {c['complaint_id']:<30} {c['question_count']} questions  budget target={c['budget_target']}")

Available complaints:
  abdominal_pain                 59 questions  budget target=35
  back_pain                      51 questions  budget target=35
  chest_pain                     60 questions  budget target=35
  constipation                   31 questions  budget target=28
  cough                          53 questions  budget target=28
  dizziness                      39 questions  budget target=35
  dysuria                        38 questions  budget target=28
  fatigue                        40 questions  budget target=28
  fever                          61 questions  budget target=35
  headache                       55 questions  budget target=35
  hematuria                      49 questions  budget target=35
  joint_pain                     46 questions  budget target=28
  loose_stool                    53 questions  budget target=28
  loss_of_consciousness          62 questions  budget target=35
  oedema                         57 questions  budget target=35
  pain            

---
## Helper

In [2]:
from pprint import pprint

all_results = []

def run_hybrid_test(complaint_id, patient_context, answers, label, checks):
    print('=' * 70)
    print(f'TEST: {label}')
    print('=' * 70)

    session = HybridIntakeSession(
        complaint_id=complaint_id,
        patient_context=patient_context,
        verbose=False,
    )

    answer_iter = iter(answers)

    while True:
        question = session.next_question()
        if question is None:
            break
        try:
            answer = next(answer_iter)
        except StopIteration:
            answer = 'No'

        result = session.submit_answer(answer)
        phase = question.get('phase', '')
        print(f"  [{phase}] {question['text']}")
        print(f"    Patient : {answer}")
        print(f"    Haiku   : {result.canonical_answer!r}  (conf: {result.confidence})")
        if result.detail_value:
            print(f"    Detail  : {result.detail_value}")

    progress = session.get_progress()
    summary  = session.get_summary()

    print()
    print(f"Questions asked : {progress['questions_asked']} / budget {progress['budget_target']}")
    print(f"Escalation      : {progress['escalation_level']}")
    print(f"Red flags       : {[f['pattern'] for f in session.red_flags]}")
    print(f"Concepts covered: {progress['covered_concept_count']}")
    print()
    print('HPI:')
    print(summary['structured']['hpi'])
    print()

    passed = []
    print('CHECKS:')
    for check_label, condition in checks(session):
        status = '\033[92m PASS\033[0m' if condition else '\033[91m FAIL\033[0m'
        print(f'  {status}  {check_label}')
        passed.append(condition)

    print()
    return session, passed

print('Helper loaded.')

Helper loaded.


---
## 1. Headache — Thunderclap / SAH scenario

In [3]:
def headache_checks(session):
    s  = session.state
    sb = session.engine.state_bool
    return [
        ('worst_headache_of_life captured as True',  sb.get('worst_headache_of_life') is True),
        ('sudden_onset captured as True',            sb.get('sudden_onset') is True),
        ('maximal_at_onset captured as True',        sb.get('maximal_at_onset') is True),
        ('neck_stiffness captured as True',          sb.get('neck_stiffness') is True),
        ('confusion_or_ams captured as True',        sb.get('confusion_or_ams') is True),
        ('severity captured',                        s.get('severity') is not None),
        ('onset captured',                           s.get('onset') is not None),
        ('red flags triggered',                      len(session.red_flags) > 0),
        ('escalation above none',                    session.escalation_level != 'none'),
    ]

headache_answers = [
    'I have this terrible headache that came on out of nowhere about two hours ago',
    'Two hours ago, really suddenly',
    'It has been going on for about two hours now',
    '10 out of 10, worst pain of my life',
    'In the back of my head and neck',
    'Yes it came on instantly like a thunderclap',
    'Yes absolutely the worst headache I have ever had',
    'Yes within the first minute it was already maximal',
    'Throbbing and pressure',
    'Constant has not changed at all',
    'Getting slightly worse if anything',
    'Nothing makes it better',
    'Any movement makes it worse',
    'No weakness that I know of',
    'No trouble speaking',
    'No vision loss',
    'No double vision',
    'I am a bit confused and not thinking straight',
    'No fever',
    'Yes my neck feels very stiff',
    'No head injury',
    'No I am not pregnant',
    'No blood thinners',
    'Yes very sensitive to light',
    'Yes sound bothers me too',
    'Yes nauseous and vomited once',
    'No aura that I noticed',
    'No this has never happened before',
    'No I do not usually get headaches',
    'Just ibuprofen which did nothing',
    'No other medications',
    'No known allergies',
]

session1, passed1 = run_hybrid_test(
    complaint_id='headache',
    patient_context={'sex': 'female', 'age': 34},
    answers=headache_answers,
    label='HEADACHE — Thunderclap / SAH scenario',
    checks=headache_checks,
)
all_results.extend(passed1)

TEST: HEADACHE — Thunderclap / SAH scenario
  [opening] Please tell me briefly what brought you in today and how this started.
    Patient : I have this terrible headache that came on out of nowhere about two hours ago
    Haiku   : 'I have this terrible headache that came on out of nowhere about two hours ago'  (conf: high)
  [core_characterize] When did the headache start? If you can, tell me whether that was minutes, hours, days, weeks, months, or years ago.
    Patient : Two hours ago, really suddenly
    Haiku   : '2 hours ago'  (conf: high)
  [core_characterize] How long has this headache been going on? If you can, tell me in minutes, hours, days, weeks, months, or years.
    Patient : It has been going on for about two hours now
    Haiku   : 'two hours'  (conf: high)
  [core_characterize] How severe is it on a scale of 0 to 10?
    Patient : 10 out of 10, worst pain of my life
    Haiku   : '10'  (conf: high)
  [early_danger_screen] Did it come on suddenly rather than building 

---
## 2. Chest Pain — ACS scenario

In [ ]:
def chest_checks(session):
    s  = session.state
    sb = session.engine.state_bool
    return [
        ('severity captured',                 s.get('severity') is not None),
        ('radiation captured as True',        sb.get('radiation') is True),
        ('diaphoresis captured as True',      sb.get('diaphoresis') is True),
        ('shortness_of_breath as True',       sb.get('shortness_of_breath') is True),
        ('nausea captured as True',           sb.get('nausea') is True),
        ('red flags triggered',               len(session.red_flags) > 0),
        ('escalation above none',             session.escalation_level != 'none'),
        ('onset captured',                    s.get('onset') is not None),
    ]

chest_answers = [
    'Chest pain that started about 45 minutes ago while I was sitting watching TV',
    '45 minutes ago',
    'About 45 minutes',
    '8 out of 10',
    'In the centre of my chest',
    'Heavy and crushing like someone sitting on me',
    'Yes it goes up into my jaw and down my left arm',
    'Constant has not come and gone',
    'Getting worse',
    'Nothing makes it better',
    'Nothing specific makes it worse',
    'Yes I am sweating a lot',
    'Yes I feel short of breath',
    'No I have not fainted',
    'Yes I feel sick to my stomach',
    'No vomiting',
    'No palpitations',
    'No cough',
    'Aspirin 75mg daily and lisinopril',
    'No allergies',
    'I had a heart attack three years ago',
    'Yes I have high blood pressure and high cholesterol',
    'I smoke about 10 cigarettes a day',
]

session2, passed2 = run_hybrid_test(
    complaint_id='chest_pain',
    patient_context={'sex': 'male', 'age': 62},
    answers=chest_answers,
    label='CHEST PAIN — ACS scenario',
    checks=chest_checks,
)
all_results.extend(passed2)

---
## 3. Seizure — Known epileptic, missed medication

In [ ]:
def seizure_checks(session):
    s  = session.state
    sb = session.engine.state_bool
    return [
        ('onset captured',                           s.get('onset') is not None),
        ('prior_seizure_history captured as True',   sb.get('prior_seizure_history') is True),
        ('medication_compliance captured as False',  sb.get('medication_compliance') is False),
        ('tongue_biting captured as True',           sb.get('tongue_biting') is True),
        ('postictal_confusion captured as True',     sb.get('postictal_confusion') is True),
        ('incontinence captured as True',            sb.get('incontinence') is True),
        ('severity captured',                        s.get('severity') is not None),
    ]

seizure_answers = [
    'I had a seizure about two hours ago',
    'About two hours ago',
    'The shaking lasted about 90 seconds',
    '7 out of 10 in terms of how distressing it was',
    'Yes I have epilepsy diagnosed five years ago',
    'Levetiracetam 500mg twice daily',
    'No I ran out three days ago and missed my doses',
    'My wife witnessed it I had no warning beforehand',
    'The shaking was in my whole body',
    'Yes I bit my tongue',
    'Yes I lost bladder control',
    'Yes confused for about 15 minutes after',
    'No weakness now that it is over',
    'No head injury',
    'No fever',
    'No I have not had alcohol or drugs',
    'I slept normally last night',
    'Just the one episode today',
    'No allergies',
]

session3, passed3 = run_hybrid_test(
    complaint_id='seizure',
    patient_context={'sex': 'male', 'age': 32},
    answers=seizure_answers,
    label='SEIZURE — Known epileptic, missed medication',
    checks=seizure_checks,
)
all_results.extend(passed3)

---
## 4. Palpitations — Exertional with presyncope

In [ ]:
def palp_checks(session):
    s  = session.state
    sb = session.engine.state_bool
    return [
        ('onset captured',                           s.get('onset') is not None),
        ('severity captured',                        s.get('severity') is not None),
        ('near-syncope captured as True',            sb.get('syncope') is True or sb.get('near_syncope') is True or sb.get('presyncope') is True),
        ('diaphoresis captured as True',             sb.get('diaphoresis') is True),
        ('chest_pain captured as False',             sb.get('chest_pain') is False),
        ('shortness_of_breath captured as False',    sb.get('shortness_of_breath') is False),
    ]

palp_answers = [
    'My heart was racing during my run this afternoon',
    'This afternoon during my run',
    'About 5 minutes',
    '7 out of 10',
    'Racing and pounding',
    'No chest pain',
    'No trouble breathing',
    'Yes I nearly fainted during the episode',
    'No it passed after I stopped exercising',
    'Yes I was sweating heavily',
    'Intermittent comes on with exercise',
    'Exercise brings it on',
    'Stopping exercise helps',
    'Dizziness during the episode',
    'No medications',
    'No allergies',
]

session4, passed4 = run_hybrid_test(
    complaint_id='palpitations',
    patient_context={'sex': 'female', 'age': 28},
    answers=palp_answers,
    label='PALPITATIONS — Exertional with presyncope',
    checks=palp_checks,
)
all_results.extend(passed4)

---
## Summary

In [ ]:
total  = len(all_results)
passed = sum(all_results)
failed = total - passed

print(f'\n==============================')
print(f'  Results: {passed}/{total} passed')
if failed:
    print(f'  FAILED:  {failed}')
else:
    print(f'  All checks passed!')
print(f'==============================')